In [1]:
# Import python modules
import os
import sys
import pandas as pd
import numpy as np

# Determine the absolute path to the src directory (one level up from notebooks)
module_path = os.path.abspath(os.path.join(os.getcwd(), "src"))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
# Import custom modules
import plotting
import utils

In [3]:
folder = os.path.join(os.path.dirname(os.getcwd()), "configs")

In [4]:
path = os.path.join(folder, "models", "config_37_v1_multi.yaml")
if not utils.load_model_config()["years"]:
    print("Yee")

Path_or_name: 
Configuration loaded from c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\base_config.yaml
Yee


In [5]:
data_folder_path = os.path.join(
    os.getcwd(), "data", "processed", "elec_s_128_ES_PT_no_bat_limit"
)
data_folder_path

'c:\\Users\\tinus\\OneDrive\\Dokumenter\\0 Master\\code\\master_project\\data\\processed\\elec_s_128_ES_PT_no_bat_limit'

In [6]:
years = [2025, 2035, 2045]
input_data = utils.load_multi_year_csv_files_from_folder(years, data_folder_path)

In [7]:
batteries = input_data["batteries"]
branches = input_data["branches"]
capacity_factors = input_data["capacity_factors"]
generators = input_data["generators"]
generator_costs = input_data["generator_costs"]
hourly_demand = input_data["hourly_demand"]
nodes = input_data["nodes"]

In [8]:
branches["exists"] = 1
# Create a copy of the dataframe for the "new" branches
branches_new = branches.copy()
# Update the index by appending " new" to the original index
branches_new.index = branches_new.index.set_levels(
    branches_new.index.levels[1].astype(str) + " new", level="line"
)
# Set the 'exists' column to 0 for the new branches
branches_new["exists"] = 0
# Concatenate the original dataframe and the new dataframe
branches = pd.concat([branches, branches_new])
# Add a new column 'exists' to the original dataframe and set it to 1
generators["exists"] = 1
# Create a copy of the dataframe for the "new" generators
generators_new = generators.copy()
# Update the index by appending " new" to the original index
generators_new.index = generators_new.index.set_levels(
    generators_new.index.levels[1].astype(str) + " new", level="generator"
)
# Set the 'exists' column to 0 for the new generators
generators_new["exists"] = 0
# Concatenate the original dataframe and the new dataframe
generators = pd.concat([generators, generators_new])
batteries["exists"] = 0

In [9]:
branches

bus0   bus1         p_max  capital_cost      length  \
year line                                                            
2025 88       ES1 0  ES1 3   2189.658631  23102.962586  449.297989   
     89       ES1 0  ES1 6   6568.975893  11729.199350  228.105190   
     90       ES1 1  ES1 2   8043.643950  16201.400425  315.078925   
     91       ES1 1  ES1 4   7775.522485  22150.028030  430.765665   
     92       ES1 1  ES1 5   1698.102612  19884.672096  386.709849   
...             ...    ...           ...           ...         ...   
2045 104 new  ES1 5  ES1 6   8758.634524  16188.029800  314.818898   
     105 new  ES1 5  ES1 7  25024.670068  12467.493385  242.463263   
     106 new  ES1 7  PT1 0   3172.770669  13131.503581  255.376691   
     107 new  ES1 7  PT1 1   1698.102612  22061.025373  429.034774   
     219 new  PT1 0  PT1 1  11707.970639  14480.882016  281.618911   

              loss_factor  exists  
year line                          
2025 88          0.017250       1  
     89          0.008758       1  
     90          0.012097       1  
     91          0.016539       1  
     92          0.014847       1  
...                   ...     ...  
2045 104 new     0.012087       0  
     105 new     0.009309       0  
     106 new     0.009805       0  
     107 new     0.016472       0  
     219 new     0.010812       0  

[114 rows x 7 columns]

In [10]:
hourly_demand

ES1 0         ES1 1        ES1 2         ES1 3        ES1 4  \
year hour                                                                      
2025 0     1540.497187   3541.798151  1849.109695   2953.965919  2331.873044   
     1     1433.747317   3296.366678  1720.974297   2749.268707  2170.284209   
     2     1335.043919   3069.435065  1602.497346   2560.000934  2020.875437   
     3     1273.019026   2926.831980  1528.046816   2441.065682  1926.987452   
     4     1250.287740   2874.569876  1500.761703   2397.477519  1892.578775   
...                ...           ...          ...           ...          ...   
2045 8755  6142.878375  14123.255438  7373.499966  11779.218761  9298.564525   
     8756  5776.361546  13280.586826  6933.557702  11076.407857  8743.762659   
     8757  5320.528871  12232.569770  6386.406674  10202.330190  8053.762096   
     8758  5093.819505  11711.336222  6114.279928   9767.605772  7710.588823   
     8759  4868.920594  11194.265141  5844.326332   9336.352978  7370.156065   

                  ES1 5        ES1 6        ES1 7         ES1 8        PT1 0  \
year hour                                                                      
2025 0      3518.575677  2092.739520  1871.996365   2787.386472  2275.245735   
     1      3274.753423  1947.721616  1742.275019   2594.232504  2182.798908   
     2      3049.309730  1813.634710  1622.331663   2415.637880  2041.494173   
     3      2907.641648  1729.374936  1546.959650   2303.409601  1916.954406   
     4      2855.722211  1698.494867  1519.336825   2262.279453  1845.104541   
...                 ...          ...          ...           ...          ...   
2045 8755  14030.653624  8344.996958  7464.762733  11114.967444  9317.490533   
     8756  13193.510129  7847.090011  7019.375245  10451.789310  8433.737190   
     8757  12152.364597  7227.848988  6465.452059   9627.002453  7965.276068   
     8758  11634.548616  6919.868127  6189.957164   9216.792927  7746.852478   
     8759  11120.867810  6614.346739  5916.662317   8809.859253  7384.729156   

                  PT1 1  
year hour                
2025 0      2474.754265  
     1      2374.201092  
     2      2220.505827  
     3      2085.045594  
     4      2006.895459  
...                 ...  
2045 8755  10134.509467  
     8756   9173.262810  
     8757   8663.723932  
     8758   8426.147522  
     8759   8032.270844  

[26280 rows x 11 columns]

In [11]:
nodes.index.names

FrozenList(['bus'])

In [12]:
"year" in batteries.index.names

True

In [13]:
generators

bus     carrier        p_nom  marginal_cost  \
year generator                                                             
2025 ES1 0 CCGT            ES1 0        CCGT  3425.300000      38.855172   
     ES1 0 coal            ES1 0        coal  1068.524933      28.196970   
     ES1 0 offwind-ac      ES1 0  offwind-ac   212.585059       0.015000   
     ES1 0 onwind          ES1 0      onwind  1487.775552       0.015000   
     ES1 0 solar           ES1 0       solar  1103.286390       0.010000   
...                          ...         ...          ...            ...   
2045 PT1 1 CCGT new        PT1 1        CCGT  3004.000000      38.892841   
     PT1 1 offwind-ac new  PT1 1  offwind-ac  3404.177205       0.015000   
     PT1 1 onwind new      PT1 1      onwind  3672.827789       0.015000   
     PT1 1 ror new         PT1 1         ror   269.600000       0.000000   
     PT1 1 solar new       PT1 1       solar   648.073599       0.010000   

                            capital_cost  co2_emissions    color  \
year generator                                                     
2025 ES1 0 CCGT             99027.729293           0.20  #b20101   
     ES1 0 coal            349976.553630           0.34  #707070   
     ES1 0 offwind-ac      183919.033054           0.00  #6895dd   
     ES1 0 onwind           96085.888020           0.00  #235ebc   
     ES1 0 solar            35602.071244           0.00  #f9d002   
...                                  ...            ...      ...   
2045 PT1 1 CCGT new         99027.729293           0.20  #b20101   
     PT1 1 offwind-ac new  185562.894661           0.00  #6895dd   
     PT1 1 onwind new       96085.888020           0.00  #235ebc   
     PT1 1 ror new         299140.224929           0.00  #4adbc8   
     PT1 1 solar new        35602.071244           0.00  #f9d002   

                                    nice_name  exists  
year generator                                         
2025 ES1 0 CCGT            Combined-Cycle Gas       1  
     ES1 0 coal                          Coal       1  
     ES1 0 offwind-ac      Offshore Wind (AC)       1  
     ES1 0 onwind                Onshore Wind       1  
     ES1 0 solar                        Solar       1  
...                                       ...     ...  
2045 PT1 1 CCGT new        Combined-Cycle Gas       0  
     PT1 1 offwind-ac new  Offshore Wind (AC)       0  
     PT1 1 onwind new            Onshore Wind       0  
     PT1 1 ror new               Run of River       0  
     PT1 1 solar new                    Solar       0  

[306 rows x 9 columns]

In [14]:
hourly_demand.loc[(2025, 0), "ES1 0"]

1540.4971869334968

In [15]:
generators

bus     carrier        p_nom  marginal_cost  \
year generator                                                             
2025 ES1 0 CCGT            ES1 0        CCGT  3425.300000      38.855172   
     ES1 0 coal            ES1 0        coal  1068.524933      28.196970   
     ES1 0 offwind-ac      ES1 0  offwind-ac   212.585059       0.015000   
     ES1 0 onwind          ES1 0      onwind  1487.775552       0.015000   
     ES1 0 solar           ES1 0       solar  1103.286390       0.010000   
...                          ...         ...          ...            ...   
2045 PT1 1 CCGT new        PT1 1        CCGT  3004.000000      38.892841   
     PT1 1 offwind-ac new  PT1 1  offwind-ac  3404.177205       0.015000   
     PT1 1 onwind new      PT1 1      onwind  3672.827789       0.015000   
     PT1 1 ror new         PT1 1         ror   269.600000       0.000000   
     PT1 1 solar new       PT1 1       solar   648.073599       0.010000   

                            capital_cost  co2_emissions    color  \
year generator                                                     
2025 ES1 0 CCGT             99027.729293           0.20  #b20101   
     ES1 0 coal            349976.553630           0.34  #707070   
     ES1 0 offwind-ac      183919.033054           0.00  #6895dd   
     ES1 0 onwind           96085.888020           0.00  #235ebc   
     ES1 0 solar            35602.071244           0.00  #f9d002   
...                                  ...            ...      ...   
2045 PT1 1 CCGT new         99027.729293           0.20  #b20101   
     PT1 1 offwind-ac new  185562.894661           0.00  #6895dd   
     PT1 1 onwind new       96085.888020           0.00  #235ebc   
     PT1 1 ror new         299140.224929           0.00  #4adbc8   
     PT1 1 solar new        35602.071244           0.00  #f9d002   

                                    nice_name  exists  
year generator                                         
2025 ES1 0 CCGT            Combined-Cycle Gas       1  
     ES1 0 coal                          Coal       1  
     ES1 0 offwind-ac      Offshore Wind (AC)       1  
     ES1 0 onwind                Onshore Wind       1  
     ES1 0 solar                        Solar       1  
...                                       ...     ...  
2045 PT1 1 CCGT new        Combined-Cycle Gas       0  
     PT1 1 offwind-ac new  Offshore Wind (AC)       0  
     PT1 1 onwind new            Onshore Wind       0  
     PT1 1 ror new               Run of River       0  
     PT1 1 solar new                    Solar       0  

[306 rows x 9 columns]

In [16]:
from models import get_model
from utils import load_model_config

model_config = load_model_config("37_v1_multi")

values = get_model(model_config)

Path_or_name: 37_v1_multi
Configuration loaded from c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\configs\models\config_37_v1_multi.yaml
Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-27


c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\src\models.py:50: PerformanceWarning: indexing past lexsort depth may impact performance.
  branches = branches.loc[y0,]
c:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\src\models.py:51: PerformanceWarning: indexing past lexsort depth may impact performance.
  generators = generators.loc[y0,]


Set parameter MIPGap to value 0.01
Set parameter BarConvTol to value 0.01
Model built in 286.21843910217285 seconds.
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: AMD Ryzen 7 5800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 1629442 rows, 1419204 columns and 4256197 nonzeros
Model fingerprint: 0xe2a584c2
Coefficient statistics:
  Matrix range     [8e-07, 1e+00]
  Objective range  [9e-04, 1e+05]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e-03, 1e+05]
Presolve removed 780753 rows and 466586 columns
Presolve time: 5.12s
Presolved: 848689 rows, 952618 columns, 2987618 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.96s

Barrier statistics:
 Dense cols : 46
 Free vars  : 26280
 AA' NZ     : 2.865e+06
 Factor NZ  : 1.290e+07 (roughly 800 MB of memor

ValueError: not enough values to unpack (expected 5, got 3)

In [ ]:
# generation_data = [(y, t, i, g[i, y, t].X) for i in G for y in Y for t in T]
generation_data

[(2025, 0, 'ES1 0 CCGT', 0.0),
 (2025, 1, 'ES1 0 CCGT', 0.0),
 (2025, 2, 'ES1 0 CCGT', 0.0),
 (2025, 3, 'ES1 0 CCGT', 0.0),
 (2025, 4, 'ES1 0 CCGT', 0.0),
 (2025, 5, 'ES1 0 CCGT', 0.0),
 (2025, 6, 'ES1 0 CCGT', 0.0),
 (2025, 7, 'ES1 0 CCGT', 0.0),
 (2025, 8, 'ES1 0 CCGT', 0.0),
 (2025, 9, 'ES1 0 CCGT', 0.0),
 (2025, 10, 'ES1 0 CCGT', 0.0),
 (2025, 11, 'ES1 0 CCGT', 0.0),
 (2025, 12, 'ES1 0 CCGT', 0.0),
 (2025, 13, 'ES1 0 CCGT', 0.0),
 (2025, 14, 'ES1 0 CCGT', 0.0),
 (2025, 15, 'ES1 0 CCGT', 0.0),
 (2025, 16, 'ES1 0 CCGT', 0.0),
 (2025, 17, 'ES1 0 CCGT', 3181.795677133754),
 (2025, 18, 'ES1 0 CCGT', 4892.4817871971845),
 (2025, 19, 'ES1 0 CCGT', 6260.750036544544),
 (2025, 20, 'ES1 0 CCGT', 6842.360302620502),
 (2025, 21, 'ES1 0 CCGT', 1506.2001785665207),
 (2025, 22, 'ES1 0 CCGT', 5919.685152732546),
 (2025, 23, 'ES1 0 CCGT', 1193.1179713358988),
 (2025, 24, 'ES1 0 CCGT', 0.0),
 (2025, 25, 'ES1 0 CCGT', 0.0),
 (2025, 26, 'ES1 0 CCGT', 0.0),
 (2025, 27, 'ES1 0 CCGT', 0.0),
 (2025, 28, '

In [ ]:
generation_df = pd.DataFrame(
    generation_data, columns=["year", "hour", "generator", "value"]
)
print(generation_df)

        year  hour        generator  value
0       2025     0       ES1 0 CCGT    0.0
1       2025     1       ES1 0 CCGT    0.0
2       2025     2       ES1 0 CCGT    0.0
3       2025     3       ES1 0 CCGT    0.0
4       2025     4       ES1 0 CCGT    0.0
...      ...   ...              ...    ...
578155  2045  8755  PT1 0 solar new    0.0
578156  2045  8756  PT1 0 solar new    0.0
578157  2045  8757  PT1 0 solar new    0.0
578158  2045  8758  PT1 0 solar new    0.0
578159  2045  8759  PT1 0 solar new    0.0

[578160 rows x 4 columns]


In [ ]:
# Helper function to reshape a variable with time and other indices
def _reshape_multi(data, index, column_name, value_name):
    """Reshape the data to have time as rows and other index (e.g., generator) as columns."""
    reshaped = data.reset_index().pivot(
        index=index, columns=column_name, values=value_name
    )
    reshaped.columns.name = None  # Remove the name of the columns for cleaner output
    return reshaped